# Automated Clinical Documentation System


Time estimate: **30** minutes

## Objectives

After completing this lab, you will be able to:

 - Understand the role of NLP in automating clinical documentation
 - Extract and summarize key information from raw clinical notes
 - Generate structured reports suitable for review by healthcare professionals

## What you will do in this lab

In this lab, you will build a basic version of an automated clinical documentation system. These systems help reduce the administrative burden on physicians by transforming raw clinical notes into readable, structured summaries.

You will:

- Load and explore sample clinical notes
- Extract relevant sections such as symptoms, diagnosis, medications, and plan
- Generate a summarized clinical report
- Structure the output in a physician-friendly format

## Overview

Automated clinical documentation systems aim to improve efficiency and reduce errors in healthcare by transforming free-text clinical narratives into structured summaries. This is a growing application of NLP in healthcare, offering value in both clinical and administrative workflows.

In this lab, you will simulate a simplified pipeline using rule-based and basic NLP techniques. The system will identify relevant sections from unstructured notes and compose a summary document — similar to what a physician might review in an electronic health record (EHR) interface.

## About the dataset

You will use a small, synthetic dataset of clinical notes. These represent fictional patient visits and include content typically seen in physician documentation.

### Dataset overview

Each note contains information on the patient's presenting symptoms, medical history, diagnosis, treatment plan, and prescribed medications. These notes are unstructured and mimic the natural flow of documentation.

### Column descriptions

1. **note_id** – Unique identifier for each note
2. **note_text** – Raw unstructured clinical note

## Setup

### Installing required libraries

You'll use standard libraries for text processing and data manipulation.

In [ ]:
# Install necessary libraries
!pip install pandas
!pip install nltk
!pip install scikit-learn

In [ ]:
# Optional: suppress warnings
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

### Importing required libraries

In [ ]:
# Import required libraries
import pandas as pd  # for data handling
import nltk  # for tokenization and NLP utilities
import re  # for pattern matching

# Download tokenizer resources
nltk.download('punkt')
nltk.download('punkt_tab')

print("Setup complete. Ready to begin!")

## Step 1: Load and inspect clinical notes

Let's start by loading a sample dataset of clinical notes and reviewing its structure. Understanding the format and content of these notes is crucial for building an effective extraction system.

In [ ]:
# Load sample clinical notes
data = {
    "note_id": [1, 2, 3, 4],
    "note_text": [
        """Patient reports chest pain and shortness of breath. PMH includes HTN. Diagnosis: angina. Plan: start nitroglycerin. Prescribed: aspirin, beta-blocker.""",
        """Complains of frequent urination and thirst. Labs indicate elevated glucose. Diagnosis: Type 2 Diabetes. Plan: initiate metformin. Medications: metformin, insulin.""",
        """Patient presents with severe headache and nausea. PMH includes migraines. Diagnosis: acute migraine. Plan: prescribe sumatriptan and recommend rest. Medications: sumatriptan 50mg, ibuprofen.""",
        """Chief complaint: persistent cough for 2 weeks. PMH includes asthma. Diagnosis: acute bronchitis. Plan: prescribe antibiotics and albuterol inhaler. Medications: azithromycin, albuterol."""
    ]
}

# Create DataFrame from the sample data
df = pd.DataFrame(data)

# Display the first few rows
print("Clinical Notes Dataset:")
print(f"Total notes: {len(df)}")
print("\nSample notes:")
df.head()

In [ ]:
# Let's examine one note in detail
print("Example Clinical Note:")
print("=" * 60)
print(df.iloc[0]['note_text'])
print("=" * 60)

## Step 2: Preprocess the clinical notes

Before extracting information, you need to clean and prepare the text. This includes normalizing the text and splitting it into sentences for easier pattern matching.

In [ ]:
def preprocess_note(note_text):
    """
    Preprocess a clinical note by tokenizing into sentences.
    We keep the original case to preserve medical abbreviations.

    Args:
        note_text: Raw clinical note text

    Returns:
        List of sentences from the note
    """
    # Tokenize the note into sentences
    sentences = nltk.sent_tokenize(note_text)

    # Remove extra whitespace from each sentence
    sentences = [sent.strip() for sent in sentences]

    return sentences

# Test preprocessing on the first note
sample_note = df.iloc[0]['note_text']
processed_sentences = preprocess_note(sample_note)

print("Preprocessed sentences:")
for i, sentence in enumerate(processed_sentences, 1):
    print(f"{i}. {sentence}")

## Step 3: Extract key information

Now you'll build functions to extract specific clinical information using pattern matching. You'll look for:
- **Symptoms**: What the patient is experiencing
- **Medical History (PMH)**: Past medical conditions
- **Diagnosis**: The identified condition
- **Plan**: The treatment approach
- **Medications**: Prescribed drugs

In [ ]:
def extract_symptoms(note_text):
    """
    Extract symptoms from clinical note.
    Looks for common symptom indicators like 'reports', 'complains of', 'presents with', etc.
    """
    symptoms = []

    # Pattern to match symptom descriptions
    patterns = [
        r'(?i)(?:patient )?(?:reports|complains of|presents with|chief complaint:)\s+([^.]+)',
        r'(?i)(?:symptoms?:)\s+([^.]+)'
    ]

    for pattern in patterns:
        matches = re.findall(pattern, note_text)
        symptoms.extend(matches)

    # Return unique symptoms, joined by semicolon
    return '; '.join(symptoms) if symptoms else "Not documented"

def extract_medical_history(note_text):
    """
    Extract past medical history (PMH) from clinical note.
    """
    # Pattern to match PMH
    match = re.search(r'(?i)PMH[\s:]+(?:includes?\s+)?([^.]+)', note_text)

    if match:
        return match.group(1).strip()
    else:
        return "Not documented"

def extract_diagnosis(note_text):
    """
    Extract diagnosis from clinical note.
    """
    # Pattern to match diagnosis
    match = re.search(r'(?i)(?:diagnosis|dx)[\s:]+([^.]+)', note_text)

    if match:
        return match.group(1).strip()
    else:
        return "Not documented"

def extract_plan(note_text):
    """
    Extract treatment plan from clinical note.
    """
    # Pattern to match plan
    match = re.search(r'(?i)(?:plan|treatment plan)[\s:]+([^.]+(?:\.[^.]*(?:prescribe|recommend|follow)[^.]*)?)', note_text)

    if match:
        return match.group(1).strip()
    else:
        return "Not documented"

def extract_medications(note_text):
    """
    Extract medications from clinical note.
    """
    medications = []

    # Patterns to match medications
    patterns = [
        r'(?i)(?:medications?|prescribed)[\s:]+([^.]+)',
        r'(?i)prescribe\s+([^.]+)'
    ]

    for pattern in patterns:
        matches = re.findall(pattern, note_text)
        medications.extend(matches)

    # Return unique medications, joined by semicolon
    return '; '.join(medications) if medications else "Not documented"

# Test extraction on the first note
test_note = df.iloc[0]['note_text']

print("Extraction Results:")
print("=" * 60)
print(f"Symptoms: {extract_symptoms(test_note)}")
print(f"Medical History: {extract_medical_history(test_note)}")
print(f"Diagnosis: {extract_diagnosis(test_note)}")
print(f"Plan: {extract_plan(test_note)}")
print(f"Medications: {extract_medications(test_note)}")
print("=" * 60)

## Step 4: Generate structured clinical summary

Now you'll combine all extraction functions into a single pipeline that creates a physician-friendly summary report.

In [ ]:
def generate_clinical_summary(note_text):
    """
    Generate a structured clinical summary from raw note text.

    Args:
        note_text: Raw clinical note

    Returns:
        Formatted string containing structured summary
    """
    # Extract all components
    symptoms = extract_symptoms(note_text)
    history = extract_medical_history(note_text)
    diagnosis = extract_diagnosis(note_text)
    plan = extract_plan(note_text)
    medications = extract_medications(note_text)

    # Format as structured report
    summary = f"""
STRUCTURED CLINICAL REPORT
{'=' * 60}

PRESENTING SYMPTOMS:
{symptoms}

PAST MEDICAL HISTORY:
{history}

DIAGNOSIS:
{diagnosis}

TREATMENT PLAN:
{plan}

MEDICATIONS:
{medications}
{'=' * 60}
    """

    return summary

# Generate summary for the first note
sample_summary = generate_clinical_summary(df.iloc[0]['note_text'])
print(sample_summary)

## Step 5: Apply the pipeline to multiple notes

Let's wrap our extraction process into a reusable function and apply it to all notes in our dataset.

In [ ]:
def process_clinical_note(note_text):
    """
    Process a clinical note and return extracted fields as a dictionary.

    Args:
        note_text: Raw clinical note text

    Returns:
        Dictionary containing extracted clinical information
    """
    return {
        'symptoms': extract_symptoms(note_text),
        'medical_history': extract_medical_history(note_text),
        'diagnosis': extract_diagnosis(note_text),
        'plan': extract_plan(note_text),
        'medications': extract_medications(note_text)
    }

# Apply processing to all notes
print("Processing all clinical notes...\n")

# Create list to store processed results
processed_results = []

for idx, row in df.iterrows():
    note_id = row['note_id']
    note_text = row['note_text']

    print(f"Processing Note {note_id}...")

    # Extract structured information
    extracted_info = process_clinical_note(note_text)

    # Add note_id to the result
    extracted_info['note_id'] = note_id

    # Append to results
    processed_results.append(extracted_info)

print("\nProcessing complete!")
print(f"Total notes processed: {len(processed_results)}")

## Step 6: Output a final summary report

Finally, let's create a structured DataFrame with all the extracted information and display it in a clean format. This represents the final output that would be stored in an EHR system or presented to physicians for review.

In [ ]:
# Create DataFrame from processed results
structured_df = pd.DataFrame(processed_results)

# Reorder columns for better readability
structured_df = structured_df[['note_id', 'symptoms', 'medical_history', 'diagnosis', 'plan', 'medications']]

# Display the structured data
print("STRUCTURED CLINICAL DOCUMENTATION")
print("=" * 80)
structured_df

In [ ]:
# Display individual formatted summaries
print("\n\nINDIVIDUAL CLINICAL SUMMARIES")
print("=" * 80)

for idx, row in df.iterrows():
    print(f"\n--- Note {row['note_id']} ---")
    print(generate_clinical_summary(row['note_text']))

In [ ]:
# Optional: Export to CSV for external use
output_file = 'structured_clinical_notes.csv'
structured_df.to_csv(output_file, index=False)
print(f"\nStructured reports exported to: {output_file}")

# Exercises

Try your hand at extracting and structuring new notes using the same pipeline.

## Exercise 1: Extract the plan section from a new note

In [ ]:
# New clinical note
new_note = "Patient presents with joint pain and swelling. PMH includes rheumatoid arthritis. Diagnosis: RA flare-up. Plan: increase prednisone dose and schedule rheumatology follow-up in 1 week. Medications: prednisone 20mg, methotrexate."

# your code goes here

<details>
    <summary>Click here for a hint</summary>

Look for keywords like `Plan:` or `Treatment plan:` using the regex pattern. Use the `extract_plan()` function we created earlier.

</details>

<details>
    <summary>Click here for solution</summary>

```python
# Extract plan from the new note
plan = extract_plan(new_note)
print("Extracted Plan:", plan)
```

</details>

## Exercise 2: Extract medications from a new note

In [ ]:
# Another clinical note
medication_note = "Chief complaint: seasonal allergies with sneezing and itchy eyes. Diagnosis: allergic rhinitis. Plan: prescribe antihistamine and nasal spray. Medications: cetirizine 10mg daily, fluticasone nasal spray."

# your code goes here

<details>
    <summary>Click here for a hint</summary>

Use the `extract_medications()` function to find all prescribed medications. Look for patterns after keywords like "Medications:" or "prescribe".

</details>

<details>
    <summary>Click here for solution</summary>

```python
# Extract medications from the note
medications = extract_medications(medication_note)
print("Extracted Medications:", medications)
```

</details>

## Exercise 3: Extract diagnosis from a new note

In [ ]:
# Clinical note with diagnosis
diagnosis_note = "Patient complains of lower back pain for 3 days. PMH includes sciatica. Diagnosis: acute lumbar strain. Plan: physical therapy and NSAIDs. Medications: ibuprofen 400mg."

# your code goes here

<details>
    <summary>Click here for a hint</summary>

The diagnosis typically follows the keyword "Diagnosis:" or "Dx:". Use the `extract_diagnosis()` function.

</details>

<details>
    <summary>Click here for solution</summary>

```python
# Extract diagnosis from the note
diagnosis = extract_diagnosis(diagnosis_note)
print("Extracted Diagnosis:", diagnosis)
```

</details>

## Exercise 4: Generate a complete structured summary for a new patient note

In [ ]:
# Complete patient note
complete_note = """Patient presents with fever, chills, and productive cough for 5 days. PMH includes COPD and smoking history.
Diagnosis: community-acquired pneumonia. Plan: initiate antibiotic therapy and chest X-ray. Recommend smoking cessation counseling.
Medications: levofloxacin 750mg daily, acetaminophen for fever."""

# your code goes here

<details>
    <summary>Click here for a hint</summary>

Use the `generate_clinical_summary()` function which combines all extraction functions to create a complete formatted report.

</details>

<details>
    <summary>Click here for solution</summary>

```python
# Generate complete structured summary
summary = generate_clinical_summary(complete_note)
print(summary)

# Alternative: Extract individual components
extracted = process_clinical_note(complete_note)
print("\nExtracted Components:")
for key, value in extracted.items():
    print(f"{key.replace('_', ' ').title()}: {value}")
```

</details>

# Congratulations!

You've completed the lab on **Automated Clinical Documentation Systems**. You learned how to extract key medical information from free-text notes and generate structured summaries that mirror real-world documentation in healthcare systems.

## Authors

Ramesh Sannareddy

Copyright © 2025 SkillUp. All rights reserved.